In [1]:
import json
import os
from datetime import datetime, timedelta
import random
from faker import Faker
import pandas as pd
from bson import ObjectId
import numpy as np

# Set fixed seed for reproducibility
random.seed(42)
np.random.seed(42)
fake = Faker()
Faker.seed(42)

# Create dataset directory
os.makedirs('Mongo_dataset', exist_ok=True)
print("Created Mongo_dataset directory")

Created Mongo_dataset directory


In [2]:
def generate_categories():
    categories = []
    category_names = [
        "Electronics", "Clothing & Accessories", "Home & Garden", "Sports & Outdoors",
        "Books & Media", "Health & Beauty", "Toys & Games", "Automotive",
        "Food & Beverages", "Office Supplies", "Jewelry", "Pet Supplies",
        "Baby & Kids", "Musical Instruments", "Crafts & Hobbies", "Industrial & Scientific"
    ]
    
    # Generate ~50 categories (16 main + subcategories)
    for i, main_cat in enumerate(category_names):
        cat_id = i + 1
        categories.append({
            "_id": ObjectId(),
            "id": cat_id,
            "name": main_cat,
            "name_lc": main_cat.lower(),
            "slug": main_cat.lower().replace(" & ", "-").replace(" ", "-"),
            "parent_id": None,
            "created_at": fake.date_time_between(start_date='-2y', end_date='now'),
            "updated_at": fake.date_time_between(start_date='-6months', end_date='now')
        })
        
        # Add 2-3 subcategories for each main category
        subcats = random.randint(2, 3)
        for j in range(subcats):
            subcat_id = len(categories) + 1
            subcat_name = f"{main_cat} - {fake.word().title()}"
            categories.append({
                "_id": ObjectId(),
                "id": subcat_id,
                "name": subcat_name,
                "name_lc": subcat_name.lower(),
                "slug": subcat_name.lower().replace(" & ", "-").replace(" ", "-").replace(" - ", "-"),
                "parent_id": cat_id,
                "created_at": fake.date_time_between(start_date='-2y', end_date='now'),
                "updated_at": fake.date_time_between(start_date='-6months', end_date='now')
            })
    
    return categories

# Generate and save categories
categories = generate_categories()
with open('Mongo_dataset/categories.ndjson', 'w') as f:
    for cat in categories:
        # Convert datetime to string for JSON serialization
        cat_copy = cat.copy()
        cat_copy['created_at'] = cat_copy['created_at'].isoformat()
        cat_copy['updated_at'] = cat_copy['updated_at'].isoformat()
        f.write(json.dumps(cat_copy, default=str) + '\n')

print(f"Generated {len(categories)} categories")

Generated 50 categories


In [3]:
def generate_customers(count=25000):
    customers = []
    
    for i in range(count):
        first_name = fake.first_name()
        last_name = fake.last_name()
        email = fake.email()
        
        customer = {
            "_id": ObjectId(),
            "id": i + 1,
            "first_name": first_name,
            "last_name": last_name,
            "email": email,
            "email_lc": email.lower(),
            "phone": fake.phone_number(),
            "address": {
                "line1": fake.street_address(),
                "line2": fake.secondary_address() if random.random() < 0.3 else None,
                "city": fake.city(),
                "state": fake.state(),
                "country": fake.country(),
                "postal_code": fake.postcode()
            },
            "is_active": random.choice([True, True, True, False]),  # 75% active
            "created_at": fake.date_time_between(start_date='-2y', end_date='now'),
            "updated_at": fake.date_time_between(start_date='-6months', end_date='now')
        }
        customers.append(customer)
    
    return customers

# Generate and save customers
customers = generate_customers()
with open('Mongo_dataset/customers.ndjson', 'w') as f:
    for customer in customers:
        customer_copy = customer.copy()
        customer_copy['created_at'] = customer_copy['created_at'].isoformat()
        customer_copy['updated_at'] = customer_copy['updated_at'].isoformat()
        f.write(json.dumps(customer_copy, default=str) + '\n')

print(f"Generated {len(customers)} customers")

Generated 25000 customers


In [4]:
def generate_products(categories, count=12000):
    products = []
    brands = ['Apple', 'Samsung', 'Nike', 'Adidas', 'Sony', 'Canon', 'Dell', 'HP', 
              'Lenovo', 'Microsoft', 'Google', 'Amazon', 'Generic', 'Premium', 'Basic']
    
    for i in range(count):
        category = random.choice(categories)
        brand = random.choice(brands)
        name = f"{brand} {fake.catch_phrase()}"
        sku = f"SKU-{i+1:06d}"
        
        product = {
            "_id": ObjectId(),
            "id": i + 1,
            "sku": sku,
            "sku_lc": sku.lower(),
            "name": name,
            "name_lc": name.lower(),
            "description": fake.text(max_nb_chars=200),
            "category_id": category['id'],
            "brand": brand,
            "brand_lc": brand.lower(),
            "price": round(random.uniform(10.0, 999.99), 2),
            "currency": "USD",
            "stock": random.randint(0, 1000),
            "popularity": random.randint(1, 10000),
            "attributes": {
                "color": random.choice(['Red', 'Blue', 'Green', 'Black', 'White', 'Gray']),
                "size": random.choice(['S', 'M', 'L', 'XL', 'XXL']) if random.random() < 0.6 else None,
                "weight": f"{random.randint(1, 50)} lbs" if random.random() < 0.4 else None,
                "material": random.choice(['Cotton', 'Plastic', 'Metal', 'Wood', 'Glass']) if random.random() < 0.5 else None
            },
            "tags": random.sample(['new', 'popular', 'sale', 'premium', 'bestseller', 'limited'], 
                                random.randint(1, 3)),
            "is_active": random.choice([True, True, True, True, False]),  # 80% active
            "created_at": fake.date_time_between(start_date='-2y', end_date='now'),
            "updated_at": fake.date_time_between(start_date='-6months', end_date='now')
        }
        products.append(product)
    
    return products

# Generate and save products
products = generate_products(categories)
with open('Mongo_dataset/products.ndjson', 'w') as f:
    for product in products:
        product_copy = product.copy()
        product_copy['created_at'] = product_copy['created_at'].isoformat()
        product_copy['updated_at'] = product_copy['updated_at'].isoformat()
        f.write(json.dumps(product_copy, default=str) + '\n')

print(f"Generated {len(products)} products")

Generated 12000 products


In [5]:
def generate_orders(customers, count=250000):
    orders = []
    statuses = ['pending', 'processing', 'shipped', 'delivered', 'cancelled']
    payment_statuses = ['pending', 'paid', 'failed', 'refunded']
    
    for i in range(count):
        customer = random.choice(customers)
        order_date = fake.date_time_between(start_date='-1y', end_date='now')
        
        # Generate amounts
        subtotal = round(random.uniform(20.0, 500.0), 2)
        tax_rate = 0.08
        tax_amount = round(subtotal * tax_rate, 2)
        shipping_fee = round(random.uniform(5.0, 15.0), 2) if random.random() < 0.8 else 0.0
        discount_amount = round(random.uniform(0.0, 50.0), 2) if random.random() < 0.3 else 0.0
        total_amount = subtotal + tax_amount + shipping_fee - discount_amount
        
        order = {
            "_id": ObjectId(),
            "id": i + 1,
            "customer_id": customer['id'],
            "order_date": order_date,
            "status": random.choice(statuses),
            "payment_status": random.choice(payment_statuses),
            "currency": "USD",
            "subtotal_amount": subtotal,
            "tax_amount": tax_amount,
            "shipping_fee": shipping_fee,
            "discount_amount": discount_amount,
            "total_amount": round(total_amount, 2),
            "coupon_code": f"COUPON{random.randint(100, 999)}" if random.random() < 0.2 else None,
            "shipping_address": customer['address'].copy(),
            "billing_address": customer['address'].copy() if random.random() < 0.8 
                             else {
                                 "line1": fake.street_address(),
                                 "city": fake.city(),
                                 "state": fake.state(),
                                 "country": fake.country(),
                                 "postal_code": fake.postcode()
                             },
            "created_at": order_date,
            "updated_at": fake.date_time_between(start_date=order_date, end_date='now')
        }
        orders.append(order)
    
    return orders

# Generate and save orders
orders = generate_orders(customers)
with open('Mongo_dataset/orders.ndjson', 'w') as f:
    for order in orders:
        order_copy = order.copy()
        order_copy['order_date'] = order_copy['order_date'].isoformat()
        order_copy['created_at'] = order_copy['created_at'].isoformat()
        order_copy['updated_at'] = order_copy['updated_at'].isoformat()
        f.write(json.dumps(order_copy, default=str) + '\n')

print(f"Generated {len(orders)} orders")

Generated 250000 orders


In [6]:
def generate_order_items(orders, products, target_count=500000):
    order_items = []
    item_id = 1
    
    for order in orders:
        # Each order has 1-4 items
        num_items = random.randint(1, 4)
        selected_products = random.sample(products, min(num_items, len(products)))
        
        for product in selected_products:
            if len(order_items) >= target_count:
                break
                
            quantity = random.randint(1, 3)
            unit_price = product['price']
            discount_amount = round(random.uniform(0.0, unit_price * 0.1), 2) if random.random() < 0.2 else 0.0
            total_price = round((unit_price * quantity) - discount_amount, 2)
            
            order_item = {
                "_id": ObjectId(),
                "id": item_id,
                "order_id": order['id'],
                "product_id": product['id'],
                "quantity": quantity,
                "unit_price": unit_price,
                "discount_amount": discount_amount,
                "total_price": total_price,
                "category_id_denorm": product['category_id'],  # Denormalized for performance
                "created_at": order['created_at']
            }
            order_items.append(order_item)
            item_id += 1
        
        if len(order_items) >= target_count:
            break
    
    return order_items

# Generate and save order_items
order_items = generate_order_items(orders, products)
with open('Mongo_dataset/order_items.ndjson', 'w') as f:
    for item in order_items:
        item_copy = item.copy()
        item_copy['created_at'] = item_copy['created_at'].isoformat() if isinstance(item_copy['created_at'], datetime) else item_copy['created_at']
        f.write(json.dumps(item_copy, default=str) + '\n')

print(f"Generated {len(order_items)} order_items")

Generated 500000 order_items


In [9]:
# Generate summary statistics
summary = {
    "dataset_info": {
        "seed": 42,
        "generation_date": datetime.now().isoformat(),
        "format": "ndjson",
        "target_database": "MongoDB Atlas"
    },
    "collections": {
        "categories": len(categories),
        "customers": len(customers),
        "products": len(products),
        "orders": len(orders),
        "order_items": len(order_items)
    },
    "total_documents": len(categories) + len(customers) + len(products) + len(orders) + len(order_items)
}

# Save summary (direct write)
with open('Mongo_dataset/dataset_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print("\nDataset Generation Complete!")
print("="*50)
for collection, count in summary["collections"].items():
    print(f"{collection}: {count:,} documents")
print(f"\nTotal documents: {summary['total_documents']:,}")
print(f"\nFiles created in 'Mongo_dataset/' directory:")
for collection in summary["collections"].keys():
    file_path = f"Mongo_dataset/{collection}.ndjson"
    if os.path.exists(file_path):
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"  - {collection}.ndjson ({size_mb:.1f} MB)")


Dataset Generation Complete!
categories: 50 documents
customers: 25,000 documents
products: 12,000 documents
orders: 250,000 documents
order_items: 500,000 documents

Total documents: 787,050

Files created in 'Mongo_dataset/' directory:
  - categories.ndjson (0.0 MB)
  - customers.ndjson (10.8 MB)
  - products.ndjson (8.1 MB)
  - orders.ndjson (173.5 MB)
  - order_items.ndjson (113.2 MB)
